# Workshop Day 1: A Quick Tutorial of the Brian2 Simulator

---
---



# Part 1: A Quick Tutorial of the Brian2 Simulator

In this workshop, we shall use a neuron simulator called Brian 2, which is based on Python and enables the instantiation of single or networks of neurons, their simulation and the plotting of elaborate results. Brian 2 is a popular simulator as well as a good representative of a large class of neuronal simulators available out there.

We assembled a small Brian tutorial], that you will go over during Day 1 of the workshop.

---
## The Brian2 Simulator

Romain Brette and Dan Goodman developed the Brian simulator for spiking neural networks as a response to the lack of hegemony between different softwares used in neural network simulation (Goodman and Brette, 2008) as each of these softwares requires learning different scripting languages. Brian presents the following advantages: 

*   As it is written on Python, it offers a tighter integration with the various tools and libraries of Python, resulting in a lot of flexibility.
*   Differential equations can be defined at the highest level using standard mathematical notation.
*   For linear differential equations, exact updates are used while for non linear differential equations, Euler and exponential Euler methods are used. 
*   Contrary to other simulators, Brian is unit-consistent. This reduces errors when modeling.
*   Further features can be easily controlled; e.g. network connectivity offers a lot of flexibility (all-to all random connectivity, specific connectivity, delays, synaptic-weight functions, among others).

The following section is a step-by-step guide of how to get started with  Brian and model a simple neuron; more information can be found [here](https://brian2.readthedocs.io/en/latest/index.html\#). In this example, the neuron is modelled by the following differential equation:

$$\frac{dV_m(t)}{dt} = \frac{I(t)}{C_m}$$

where $V_m$ is the membrane potential, $C_m$ is the membrane capacitance and $I$ is the current input.  


---
## Importing the Brian Toolbox
First, every Brian script in Python begins by importing the Brian toolbox. Other toolboxes can also be imported, such as *matplotlib*  for plotting tools and *numpy* to help with mathematical computations. After importing the libraries, it is useful to invoke the *start_scope* function as it starts a new scope for the magic functions. In other words, it resets the simulations by clearing any preceding neurongroup defininitions.


## Run This First

Run the next code cell before the rest of the notebook.

- In Google Colab it installs any missing notebook-only packages and enables widget support.
- In local JupyterLab it only verifies imports against your active environment.
- Local setup: create a virtual environment and install the packages in `requirements-notebooks.txt`.


In [ ]:
# Notebook runtime setup for Google Colab and local JupyterLab.
import importlib
import subprocess
import sys

try:
    from google.colab import output as colab_output
    IS_COLAB = True
except ImportError:
    colab_output = None
    IS_COLAB = False


def ensure_notebook_packages(requirements):
    if not IS_COLAB:
        return

    missing = []
    for package_name, module_name in requirements:
        if importlib.util.find_spec(module_name) is None:
            missing.append(package_name)

    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])


NOTEBOOK_REQUIREMENTS = [('brian2', 'brian2')]
ensure_notebook_packages(NOTEBOOK_REQUIREMENTS)

if IS_COLAB:
    colab_output.enable_custom_widget_manager()


In [ ]:
%%time


from brian2 import *
import matplotlib.pyplot as plt # for plotting in python
import brian2.numpy_ as np
import matplotlib.gridspec as gridspec

start_scope() # resets the simulations and clears any preceding model definitions


<font color=green>*Hint: Let your mouse pointer rest on top of the various Python or Brian functions, modules etc. and a quick reference entry should pop up.*</font>

## Notebook magic commands

In the above code cell, preceding the Python and Brian commands, you may have noticed that there is a ```%%time``` command, specially highlighted. This is known as a Notebook *magic command* which help add more functionality to the code cell than the "user code", in this case Python and Brian.

Specifically ```%%time``` measures the time it takes to execute the whole code cell and separates the time between so-called CPU (pure compute) and Wall-clock (human-perceptible) time.

You can read more about magic commands that you can try in this tutorial and in the next assignments [here](https://ipython.readthedocs.io/en/stable/interactive/magics.html#cell-magics).

The magic command ```%%timeit``` can also be used. The notable difference of this command is that it runs the specified code many times and computes an average. You can specify the number of runs with the ```-n``` option, but if nothing is passed a fitting value will be chosen based on computation time. More advanced magic-command uses can be found [here](https://www.dataquest.io/blog/advanced-jupyter-notebooks-tutorial/).

Keep in mind that a single-percent symbol ```%``` will make any magic command run for a single line of the code, while a double-percent symbol ```%%``` will make the command run for the whole code cell.


> <font color=yellow>QC1.1: As you may have noticed, CPU and Wall-clock times are never exactly the same. Can you imagine why that is?</font>

> <font color=yellow>QC1.2: Try running the same code cell a few times. Do you get the same time measurement each time? Why (not)?</font>
 

---
## Defining Parameters and Dealing with Units
Second, the parameters of the neurons are defined with the correct units. Brian accepts all basic SI units accompanied by all standard prefixes. 

In [ ]:
%%time

Cm = 1.0*uF/cm**2  #The membrane capacitance


---
## Defining an Equation
Equations are defined **as strings** using standard mathematical notations; units also have to be defined as shown below. In this example, the voltage of the membrane $V$ is being computed, thus the unit is in Volts. The input for this example is the current $I$ and thus, is defined as shown below (with unit amp/m$^2$). Note that Brian recommends using triple single quotes when definining equations.

In [ ]:
%%time

eqs_IF = '''
    dV/dt = I/Cm : volt
    I : amp*meter**-2
'''


---
## Creating a Neuron
Neuron models are generated using the ```NeuronGroup()``` function, which requires the specification of the number of neurons *N*, the neuron equation *eqs* as well as the integration *method*, for example ['forward Euler'](https://en.wikipedia.org/wiki/Euler_method).

In [ ]:
%%time

N_IF = 1
Neuron_IF = NeuronGroup(N_IF, eqs_IF, method = 'euler')


---
## Recording
At this point, the objective of the simulation must be specified through the ```StateMonitor()``` function for monitoring during simulation. For this example, both the membrane potential $V$ and the input current $I$ are measured at every simulation step and saved in the ```Neuron_statemon``` for later usage.

In [ ]:
%%time

Neuron_statemon_IF = StateMonitor(Neuron_IF, variables=['V','I'], record = True) 
  # "record = True" can also give "record = [0, 10] where the numbers are the indices of neurons to be recorded.
store('IF_Neuron') # stores the neuron and state monitor

# quick validation
Neuron_validation_IF = Neuron_IF.equations # return LaTeX code that can be copied and rendered below for reporting but also for validation purposes.
f"{Neuron_validation_IF}" # use a simple Python f-string to print the LaTeX math formula inline 


---
## Simulating a Neuron
To start the simulation, the runtime (100 milliseconds in this example) and the input variables (applied current in this example) are defined. Then, the results can be plotted. The figure below shows the voltage response of this neuron. As expected, the membrane potential increases linearly with a constant current. 


In [ ]:
%%time

restore('IF_Neuron')
# We set the experiment runtime and initial input
runtime = 100*ms # units are added via the multiplication symbol
Neuron_IF.I = 1*uamp*cm**-2 # remember "I" is a variable contained in object "Neuron"
run(runtime) # run the neuron 

# Plot
fig = plt.figure(figsize=(15, 6))

gs1 = gridspec.GridSpec(10, 5)
gs1.update(left=0.01, right=0.5, wspace=9)
ax1 = plt.subplot(gs1[:-1, :])
ax1.plot(Neuron_statemon_IF.t/ms, Neuron_statemon_IF.V[0]/mvolt,'C1',lw='2')
ylabel('Voltage (mV)')

ax2 = plt.subplot(gs1[-1, :])
ax2.plot(Neuron_statemon_IF.t/ms, Neuron_statemon_IF.I[0]/(uamp/cm**2),'C2',lw='2')
xlabel('t (msec)')
ylabel(u"Current (\u03bcamp/cm\u00B2)")

show()


> <font color=yellow>QC1.3: Try running the same simulation for runtimes of 100 ms (original), 1000 ms and 10000 ms. Plot CPU and Wall-clock times as a function of the different runtimes. What do you observe? Why?</font>

<font color=green>*Hint: These are known as performance-scalability plots and you will be asked to plot a few of them throughout this workshop. You can plot directly in a notebook via matplot lib or e.g. through Excel in your computer using Line plots or Scatter plots with Markers.*</font>

---
## Simulating Different Currents Over Time
To see the response of the membrane potential to a varying current input, it is only needed to change the ```Neuron.I``` value and run the code again. For instance, a 50-millisecond current pulse can be created as shown below. 

In [ ]:
%%time

restore('IF_Neuron')

# We run the code in blocks:
# First choose a runtime for the first part
runtime = 100*ms # Choose a runtime
Neuron_IF.I = 0*uamp*cm**-2 # set a new input current
run(runtime) # run the neuron for the specified runtime
# Second block:
Neuron_IF.I = 1*uamp*cm**-2 # set a new input current
run(50*ms) # run the neuron
# Third block:
Neuron_IF.I = 0*uamp*cm**-2 # set a new input current
run(100*ms) # run the neuron

# Plot
fig = plt.figure(figsize=(15, 6))

gs1 = gridspec.GridSpec(10, 5)
gs1.update(left=0.01, right=0.5, wspace=9)
ax1 = plt.subplot(gs1[:-1, :])
ax1.plot(Neuron_statemon_IF.t/ms, Neuron_statemon_IF.V[0]/mvolt,'C1',lw='2')
ylabel('V mV')

ax2 = plt.subplot(gs1[-1, :])
ax2.plot(Neuron_statemon_IF.t/ms, Neuron_statemon_IF.I[0]/(uamp/cm**2),'C2',lw='2')
xlabel('t (msec)')
ylabel('I (\u03bcamp/cm\u00B2)')

show()


By implementing a threshold potential at which a spike occurs, the neuron model in this example becomes a spiking-neuron model known as the **Integrate-and-Fire model** (IaF or I&F). It is the basis for modern spiking-neuron models which will be discussed next. 

---
## Leaky Integrate-and-Fire Model
This model reflects the diffusion of ions occurring through the membrane while the cell is not in equilibrium (Dayan and Abbott, 2005). This is achieved by introducing a leak term $I_l(t) = \frac{V_m(t)}{R_m}$. The introduction of this leak term causes the cell to fire only when $I(t)$ is bigger than a new threshold $I_{th} = \frac{V_{th}}{R_m}$ (Izhikevich,  2007). Hence, if the current does not reach this new threshold current, the leak term will leak out any change in potential. 
$$ C_m\frac{dV_m(t)}{dt} = I(t)-\frac{V_m(t)}{R_m}$$
where $R_m$ is the membrane resistance.


---
## Implementing the Leaky I&F Model on the Brian2 Simulator
The membrane resistance parameter needs to be defined as well as the new leak current $I_l$. This model can be improved by including a refractory period that prevents the neuron to fire for a certain amount of time after the spike. This period can be manually added to the *NeuronGroup* function as follows:

In [ ]:
%%time

start_scope()

Cm = 1.0*uF/cm**2   
Vt = -40*mvolt      #Threshold
Vr = -65*mvolt      #Reset Voltage
Rm =  10*Mohm*cm**2 #Membrane resistance
eqs_LIF = '''
    dV/dt = (I-I_l)/Cm : volt
    I_l = V/Rm : amp*meter**-2
    I : amp*meter**-2
'''
N = 1
Neuron_LIF = NeuronGroup(N, eqs_LIF, threshold='V>Vt', reset='V=Vr', refractory = 2*ms, method = 'euler', name='IF_Neuron')
Neuron_statemon_LIF = StateMonitor(Neuron_LIF, variables=['V','I'], record = True)


Neuron_LIF.V = -70*mvolt
Neuron_LIF.I = 0*uamp*cm**-2
run(100*ms)

Neuron_LIF.I = 1*uamp*cm**-2
run(50*ms)

Neuron_LIF.I = 0*uamp*cm**-2
run(100*ms)

Neuron_LIF.I = 1*uamp*cm**-2
run(100*ms)

Neuron_LIF.I = 0*uamp*cm**-2
run(50*ms)

Neuron_LIF.I = 1*uamp*cm**-2
run(50*ms)

Neuron_LIF.I = 0*uamp*cm**-2
run(runtime)

# Plot
fig = plt.figure(figsize=(15, 6)) 

gs1 = gridspec.GridSpec(10, 5)
gs1.update(left=0.01, right=0.5, wspace=9)
ax1 = plt.subplot(gs1[:-1, :])
ax1.plot(Neuron_statemon_LIF.t/ms, Neuron_statemon_LIF.V[0]/mvolt,'C1',lw='2')
ylabel('V mV')

ax2 = plt.subplot(gs1[-1, :])
ax2.plot(Neuron_statemon_LIF.t/ms, Neuron_statemon_LIF.I[0]/(uamp/cm**2),'C2',lw='2')
xlabel('t (msec)')
ylabel('I (\u03bcamp/cm\u00B2)')

show()


In tomorrow's lab, we will occupy ourselves with single-neuron simulations and the respective increases in computational effort in response to introducing advanced modeling features. In view of Brian's multithreading capabilities still not working as intended, we have resorted to the underlying libraries of Python to demonstrate multithreading execution and conduct some performance experiments.

Stay tuned and see you tomorrow!